# 🧠 Tutorial 05: Meta-Learning con Memoria (RNN-based)

## Aprendiendo a Optimizar con Redes Recurrentes

En este tutorial aprenderás:

- 🔄 Cómo usar RNNs/LSTMs como "optimizadores aprendidos"
- 🧠 Meta-Learning como proceso secuencial
- 💻 Implementación de un optimizador basado en LSTM
- 📊 Comparación con optimizadores tradicionales

---

## 📖 Teoría: RNNs como Optimizadores

### La Idea Central:

En lugar de usar una regla fija de actualización (como SGD o Adam), **aprende** la regla de actualización con una RNN.

### Proceso:

1. La RNN recibe el gradiente actual como input
2. Mantiene un estado oculto que "recuerda" gradientes anteriores  
3. Produce una actualización de parámetros como output

### Matemáticamente:

$$h_t = \text{LSTM}(g_t, h_{t-1})$$
$$\theta_t = \theta_{t-1} + f(h_t)$$

donde $g_t$ es el gradiente en el paso $t$.

---


## 📑 Table of Contents- [1 - Introduction to Memory-Augmented Meta-Learning](#1)    - [1.1 - Why Memory for Meta-Learning?](#1-1)    - [1.2 - Types of Memory Architectures](#1-2)- [2 - Setup](#2)- [3 - Neural Turing Machines Background](#3)- [4 - Exercise 1 - Memory Module](#ex-1)- [5 - Exercise 2 - Memory-Augmented Network](#ex-2)- [6 - Exercise 3 - Meta-Learning with Memory](#ex-3)- [7 - Training Loop](#7)- [8 - Evaluation](#8)- [9 - Memory Visualization](#9)- [10 - Comparison: With vs Without Memory](#10)- [11 - Use Cases and Applications](#11)- [12 - Summary](#12)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('..')

from utils.test_utils import print_success, print_hint, HintSystem
from utils.data_utils import create_sine_task, set_seed

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Setup completo!")

<a name='1-1'></a>### 1.1 - Why Memory for Meta-Learning?Traditional neural networks are **stateless** - they process each input independently.**Problem for Few-Shot Learning:**- Need to remember support examples during query prediction- Standard approaches: embed in weights (slow) or attention (limited capacity)**Memory-Augmented Solution:**- External memory bank to store and retrieve information- Fast updates without changing weights- Explicit separation: what to remember vs how to process<table><tr>    <td><b>Approach</b></td>    <td><b>How it Remembers</b></td>    <td><b>Speed</b></td>    <td><b>Capacity</b></td></tr><tr>    <td>Weight Updates</td>    <td>Gradient descent</td>    <td>Slow (backprop)</td>    <td>Limited by params</td></tr><tr>    <td>Attention</td>    <td>Attention weights</td>    <td>Fast (forward pass)</td>    <td>Limited by sequence</td></tr><tr>    <td>External Memory</td>    <td>Read/write to memory</td>    <td>Fast (addressable)</td>    <td>Large (expandable)</td></tr></table>**Key Insight**: Memory allows **rapid integration** of new information without slow gradient updates!

<a name='1-2'></a>### 1.2 - Types of Memory Architectures**1. Neural Turing Machines (NTM)**- Differentiable memory with read/write heads- Content-based and location-based addressing- Used in: sequence tasks, copy tasks**2. Differentiable Neural Computer (DNC)**- Extension of NTM with better memory management- Temporal linkage between memory slots- Used in: reasoning, graph traversal**3. Memory Networks**- Simple key-value memory store- Attention-based retrieval- Used in: question answering, few-shot learning**4. MANN (Memory-Augmented Neural Networks)**- Specialized for one-shot learning- LSTM controller + external memory- Used in: few-shot classification**For this tutorial, we'll implement a simplified MANN-style architecture.**

<a name='3'></a>## 3 - Neural Turing Machines BackgroundA Neural Turing Machine has three key components:**1. Controller**: Neural network (LSTM/FF) that processes input**2. Memory Matrix**: M ∈ ℝ^(N×M) where N=slots, M=memory size**3. Read/Write Heads**: Mechanisms to access memory### Memory Operations:**Read Operation:**$$r_t = \sum_{i=1}^N w_t^r(i) M_t(i)$$where $w_t^r$ is the read attention weight vector.**Write Operation:**$$M_t(i) = M_{t-1}(i) [1 - w_t^w(i) e_t] + w_t^w(i) a_t$$where:- $e_t$ is erase vector (what to forget)- $a_t$ is add vector (what to write)- $w_t^w$ is write attention weights### Addressing:**Content-Based**:$$w_t^c(i) = \frac{\exp(\beta_t K(k_t, M_t(i)))}{\sum_j \exp(\beta_t K(k_t, M_t(j)))}$$where $K$ is cosine similarity between key $k_t$ and memory row $M_t(i)$.**Simplified for our tutorial**: We'll use content-based addressing only.

---

## 💻 Ejercicio: Optimizador LSTM

**Tu tarea**: Completa el optimizador basado en LSTM.

In [ ]:
class LSTMOptimizer(nn.Module):
    """
    Optimizador aprendido usando LSTM.
    """
    
    def __init__(self, input_size=1, hidden_size=20):
        super(LSTMOptimizer, self).__init__()
        
        # TODO: Define una LSTM que tome gradientes como input
        # Input: gradiente (1D), Hidden state: información histórica
        # Output: actualización de parámetros
        
        self.lstm = None  # TODO: nn.LSTM(input_size, hidden_size)
        self.fc = None    # TODO: nn.Linear(hidden_size, 1) para generar actualizaciones
        
    def forward(self, gradients, hidden=None):
        """
        Args:
            gradients: [seq_len, batch, 1] - Gradientes
            hidden: Estado oculto de LSTM
        
        Returns:
            updates: [seq_len, batch, 1] - Actualizaciones de parámetros
            hidden: Nuevo estado oculto
        """
        # TODO: Pasa gradientes por LSTM y genera actualizaciones
        pass


# Sistema de pistas
hints_lstm_opt = HintSystem([
    "LSTM toma (input_size, hidden_size) y devuelve output y (h, c).",
    "Usa self.lstm(gradients, hidden) para procesar la secuencia.",
    "Aplica self.fc() al output de LSTM para generar actualizaciones.",
    "Código: out, hidden = self.lstm(gradients, hidden); updates = self.fc(out); return updates, hidden"
])

# Para ver pistas
hints_lstm_opt.show_hint()

---

## 📊 Comparación: LSTM vs SGD

En este tutorial conceptual, la idea es que el optimizador LSTM puede aprender estrategias de actualización más sofisticadas que reglas fijas como SGD o Adam.

### Ventajas del LSTM Optimizer:

- ✅ Aprende de la historia de gradientes
- ✅ Puede adaptarse a la geometría del problema
- ✅ No requiere tuning de hyperparámetros

### Limitaciones:

- ⚠️ Requiere meta-training extenso
- ⚠️ Puede no generalizar a arquitecturas muy diferentes
- ⚠️ Computacionalmente costoso

---

## 🎓 Conclusión

Has aprendido cómo las RNNs pueden actuar como optimizadores aprendidos, utilizando memoria para tomar mejores decisiones de actualización.

### 🚀 Próximo Tutorial:

En el **Tutorial 06** llevaremos Meta-Learning al mundo de **Reinforcement Learning**, donde la adaptación rápida es crucial.


<a name='ex-1'></a>## 4 - Exercise 1 - Memory ModuleImplement a basic external memory module with read/write operations.

In [ ]:
class ExternalMemory(nn.Module):    """    External memory module for meta-learning.        Args:        memory_size: Number of memory slots (N)        memory_dim: Dimension of each memory slot (M)    """        def __init__(self, memory_size=128, memory_dim=64):        super(ExternalMemory, self).__init__()                self.memory_size = memory_size        self.memory_dim = memory_dim                # Initialize memory to zeros        self.register_buffer('memory', torch.zeros(memory_size, memory_dim))            def read(self, query, temperature=1.0):        """        Read from memory using content-based addressing.                Args:            query: [batch, memory_dim] - query vector            temperature: softmax temperature                Returns:            read_vector: [batch, memory_dim] - retrieved from memory            attention: [batch, memory_size] - attention weights        """        # TODO: Implement content-based read        # 1. Compute similarity between query and all memory slots        # 2. Apply softmax to get attention weights        # 3. Weighted sum of memory slots                pass  # TODO        def write(self, key, value, erase_strength=0.0):        """        Write to memory using content-based addressing.                Args:            key: [batch, memory_dim] - where to write            value: [batch, memory_dim] - what to write            erase_strength: how much to erase first (0-1)                Returns:            attention: [batch, memory_size] - write attention weights        """        # TODO: Implement content-based write        # 1. Find where to write using key        # 2. Optionally erase        # 3. Add new value                pass  # TODO# Hintshints_memory = HintSystem([    "For read: use F.cosine_similarity or torch.matmul with normalized vectors.",    "Attention weights: F.softmax(similarities / temperature, dim=1).",    "Read vector: torch.matmul(attention, self.memory).",    "For write: find attention weights same as read, then update memory in-place.",    "Write update: memory = memory * (1 - erase) + attention.T @ value"])hints_memory.show_hint()

<a name='10'></a>## 10 - Comparison: With vs Without MemoryLet's compare a model with external memory vs standard approach.

<a name='11'></a>## 11 - Use Cases and ApplicationsMemory-augmented meta-learning excels in specific scenarios:### 1. **Continual Learning**- **Problem**: Learn new tasks without forgetting old ones- **Memory Solution**: Store task-specific information in memory- **Example**: Robot learning new skills incrementally### 2. **Personalization**- **Problem**: Adapt to individual users quickly- **Memory Solution**: Store user-specific preferences- **Example**: Personalized recommendations, adaptive UI### 3. **Dialogue Systems**- **Problem**: Remember context across conversation- **Memory Solution**: External memory for conversation history- **Example**: Customer service bots, virtual assistants### 4. **Reasoning Tasks**- **Problem**: Multi-hop reasoning over facts- **Memory Solution**: Store and retrieve relevant facts- **Example**: Question answering, theorem proving### 5. **Program Synthesis**- **Problem**: Generate code from few examples- **Memory Solution**: Store code patterns and templates- **Example**: Code completion, auto-programming### When to Use Memory-Augmented Approaches:✅ **Good for:**- Tasks requiring explicit storage- Sequential decision making- Need for interpretability (can inspect memory)- Continual/lifelong learning⚠️ **Consider alternatives if:**- Simple classification (Prototypical Networks simpler)- No sequential structure (standard methods sufficient)- Very limited computational resources

In [ ]:
print("🔬 Comparing Memory-Augmented vs Standard Model...\n")# Model WITH memorymodel_with_memory = MemoryAugmentedNetwork(    input_dim=784,    hidden_dim=128,    memory_size=64,    memory_dim=32)# Model WITHOUT memory (standard LSTM)model_without_memory = nn.LSTM(    input_size=784,    hidden_size=128,    num_layers=2)# Train both on same few-shot tasksresults = {'with_memory': [], 'without_memory': []}for scenario in ['5-way 1-shot', '5-way 5-shot', '10-way 1-shot']:    print(f"Testing: {scenario}")        # ... train and evaluate both models ...        results['with_memory'].append(acc_mem)    results['without_memory'].append(acc_standard)        print(f"  With Memory: {acc_mem:.2%}")    print(f"  Without Memory: {acc_standard:.2%}")    print(f"  Improvement: +{(acc_mem - acc_standard):.2%}\n")# Visualizescenarios_list = ['5-way 1-shot', '5-way 5-shot', '10-way 1-shot']fig, ax = plt.subplots(figsize=(12, 6))x = np.arange(len(scenarios_list))width = 0.35ax.bar(x - width/2, results['without_memory'], width,        label='Standard LSTM', color='coral', alpha=0.8)ax.bar(x + width/2, results['with_memory'], width,       label='Memory-Augmented', color='steelblue', alpha=0.8)ax.set_xlabel('Scenario', fontsize=12)ax.set_ylabel('Accuracy', fontsize=12)ax.set_title('Memory-Augmented vs Standard Model', fontsize=14)ax.set_xticks(x)ax.set_xticklabels(scenarios_list)ax.legend()ax.grid(True, alpha=0.3, axis='y')plt.tight_layout()plt.show()print("\n📊 Key Findings:")print("  - Memory helps most in 1-shot scenarios")print("  - Provides explicit storage for support examples")print("  - More interpretable (can visualize memory access)")

<a name='12'></a>## 12 - Summary and Conclusions<font color='blue'>**What you should remember:**- ✅ External memory provides **fast, explicit storage** for meta-learning- ✅ Key operations: **content-based read** and **write**- ✅ Enables **rapid integration** of new information without gradient updates- ✅ Particularly useful for **sequential** and **continual learning** tasks- ✅ More complex than metric-based methods but adds powerful capabilities- ✅ Can be combined with other meta-learning approaches</font>### Comparison with Other Methods:<table><tr>    <td><b>Method</b></td>    <td><b>Adaptation Mechanism</b></td>    <td><b>Best For</b></td></tr><tr>    <td>Prototypical Networks</td>    <td>Distance in embedding space</td>    <td>Simple classification</td></tr><tr>    <td>MAML</td>    <td>Gradient descent steps</td>    <td>General tasks</td></tr><tr>    <td>Memory-Augmented</td>    <td>External memory read/write</td>    <td>Sequential, continual learning</td></tr></table>### Key Takeaways:1. **Trade-off**: More complex architecture for additional capabilities2. **When to use**: Sequential tasks, need for explicit storage3. **Implementation**: Can build on NTM, DNC, or custom memory4. **Performance**: Competitive with other methods, excels in specific domains### 📚 References:- Neural Turing Machines: [Graves et al., 2014](https://arxiv.org/abs/1410.5401)- MANN: [Santoro et al., 2016](https://arxiv.org/abs/1605.06065)- DNC: [Graves et al., 2016](https://www.nature.com/articles/nature20101)---## 🎉 Congratulations!You now understand **memory-augmented meta-learning**, a powerful extension for sequential and continual learning tasks!**Next**: Tutorial 06 - Meta-RL applies meta-learning to reinforcement learning! 🤖